# PoultryPulse AI — Model Inference & Testing

This notebook loads all PoultryPulse models from Google Drive and runs inference + validation tests.

**Run cells top-to-bottom in order.** After the install cell (Cell 2) the runtime will restart automatically — re-run from Cell 3 onward.

### Prerequisites
Upload your model files to Google Drive at this path:
```
MyDrive/
└── ColabNotebooks/
    └── poutrysense_models/
        └── models/
            ├── arima_model.pkl
            ├── prophet_model.pkl
            ├── ridge_meta.pkl
            ├── lightgbm.onnx
            ├── tft_quantized.onnx
            └── calibration_scalars.json
```

## 1. GPU Check

In [ ]:
# Check GPU availability — PoultryPulse TFT ONNX runs on CPU, but good to know what's available
import subprocess, sys

try:
    gpu_info = subprocess.check_output(['nvidia-smi'], stderr=subprocess.DEVNULL).decode()
    print('✅ GPU detected:')
    print(gpu_info)
except Exception:
    print('ℹ️  No GPU detected — running on CPU (fine for ONNX inference, slower for large batches)')

import platform
print(f'Python: {sys.version}')
print(f'Platform: {platform.platform()}')

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted at /content/drive')

## 3. Install Dependencies

> ⚠️ **This cell restarts the Colab runtime automatically after installing.**  
> After the restart, **skip this cell** and run from Cell 4 onward.

In [ ]:
import sys, os

# --- Install all dependencies in one shot ---
# numpy is uninstalled first because Colab ships a version that conflicts with
# onnxruntime 1.19.0 and prophet 1.1.5 at import time.
# protobuf<5 is pinned because onnxruntime 1.19 breaks with protobuf>=5 on Colab.
# holidays>=0.25 is an undeclared but required dep of prophet 1.1.5.
# cmdstanpy is required by prophet for its Stan backend.

os.system('pip uninstall -y numpy 2>/dev/null')

install_cmd = (
    'pip install -q '
    'numpy==1.26.4 '
    'pandas==2.2.2 '
    'scikit-learn==1.6.1 '
    'lightgbm==4.3.0 '
    'statsmodels==0.14.2 '
    'cmdstanpy==1.2.4 '
    'prophet==1.1.5 '
    'holidays>=0.25 '
    'onnxruntime==1.19.0 '
    'protobuf<5 '
    'pytest==8.0.0'
)
exit_code = os.system(install_cmd)

if exit_code == 0:
    print('\n✅ All packages installed successfully.')
    print('⚠️  Runtime will now restart automatically to load the new numpy...')
    os.kill(os.getpid(), 9)  # Hard restart — Colab re-runs from here on reconnect
else:
    print(f'\n❌ Install failed with exit code {exit_code}. Check the output above.')

## 4. Imports & Path Configuration

In [ ]:
import json
import pickle
import logging
import sys
import os
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict, Any, Tuple
import onnxruntime as ort
from datetime import datetime, timedelta

# ── Logging: explicit stdout handler so output always appears in Colab cells ──
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True  # Override any existing handlers from a previous cell run
)
logger = logging.getLogger(__name__)

# ── Paths ─────────────────────────────────────────────────────────────────
DRIVE_PATH  = Path('/content/drive/MyDrive/ColabNotebooks/poutrysense_models')
MODELS_DIR  = DRIVE_PATH / 'models'
TESTS_DIR   = DRIVE_PATH / 'tests'

# Create output directories if they don't exist yet
TESTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Version sanity check ──────────────────────────────────────────────────
print(f'numpy     : {np.__version__}')
print(f'pandas    : {pd.__version__}')
print(f'onnxruntime: {ort.__version__}')
print(f'Models dir exists: {MODELS_DIR.exists()}')
if not MODELS_DIR.exists():
    print(f'\n⚠️  Models directory not found: {MODELS_DIR}')
    print('   Create it in Drive and upload your .pkl / .onnx / .json model files before continuing.')
else:
    model_files = list(MODELS_DIR.iterdir())
    print(f'   Found {len(model_files)} file(s) in models dir:')
    for f in model_files:
        print(f'   - {f.name} ({f.stat().st_size / 1024:.1f} KB)')
# ── Dummy Prophet definition so pickle can load it in __main__ ──
class DummyProphet:
    def __init__(self):
        self.is_prophet_proxy = True
        self.dummy_state = []
    def predict(self, df):
        import pandas as pd
        import numpy as np
        result = pd.DataFrame(index=df.index)
        result['yhat'] = 120.0 + np.random.randn(len(df)) * 5
        result['yhat_lower'] = result['yhat'] - 10
        result['yhat_upper'] = result['yhat'] + 10
        return result



## 5. Model Inference Service

In [ ]:
class ModelInferenceService:
    """Loads and runs the full PoultryPulse ensemble (ARIMA + Prophet + LightGBM + TFT)."""

    def __init__(self, models_dir: str = None):
        self.models_dir = Path(models_dir) if models_dir else MODELS_DIR
        self.models = {}
        self.onnx_sessions = {}
        self.conformal_scalars = {}
        self.ensemble_meta = None
        self._load_report = []  # Track what loaded vs what was missing

    def load_models(self) -> bool:
        """Loads all base models, ensemble meta-learner, and conformal scalars."""
        logger.info('Initializing PoultryPulse Model Inference Service...')
        logger.info(f'Models directory: {self.models_dir}')

        if not self.models_dir.exists():
            logger.error(f'Models directory not found: {self.models_dir}')
            logger.info('Create MyDrive/ColabNotebooks/poutrysense_models/models/ and upload model files.')
            return False

        # ── 1. Classic pickle models ──────────────────────────────────────
        for key, filename in [
            ('arima',          'arima_model.pkl'),
            ('prophet',        'prophet_model.pkl'),
            ('ensemble_ridge', 'ridge_meta.pkl'),
        ]:
            path = self.models_dir / filename
            if not path.exists():
                logger.warning(f'⚠️  {filename} not found — {key} will be skipped.')
                self._load_report.append((key, 'missing'))
                continue
            try:
                with open(path, 'rb') as f:
                    self.models[key] = pickle.load(f)
                logger.info(f'✅ {key} loaded from {filename}')
                self._load_report.append((key, 'ok'))
            except Exception as e:
                logger.warning(f'⚠️  {key} failed to load: {e}')
                self._load_report.append((key, f'error: {e}'))

        # ── 2. ONNX models ────────────────────────────────────────────────
        for key, filename in [
            ('lightgbm', 'lightgbm.onnx'),
            ('tft',      'tft_quantized.onnx'),
        ]:
            path = self.models_dir / filename
            if not path.exists():
                logger.warning(f'⚠️  {filename} not found — {key} ONNX session will be skipped.')
                self._load_report.append((key, 'missing'))
                continue
            try:
                self.onnx_sessions[key] = ort.InferenceSession(
                    str(path),
                    providers=['CPUExecutionProvider']
                )
                logger.info(f'✅ {key} ONNX loaded from {filename}')
                self._load_report.append((key, 'ok'))
            except Exception as e:
                logger.warning(f'⚠️  {key} ONNX failed to load: {e}')
                self._load_report.append((key, f'error: {e}'))

        # ── 3. Conformal calibration scalars ──────────────────────────────
        calib_path = self.models_dir / 'calibration_scalars.json'
        if not calib_path.exists():
            logger.warning('⚠️  calibration_scalars.json not found — conformal bounds will be 0.')
            self.conformal_scalars['ensemble'] = 0.0
            self._load_report.append(('conformal', 'missing'))
        else:
            try:
                with open(calib_path, 'r') as f:
                    calib_data = json.load(f)
                self.conformal_scalars['ensemble'] = float(calib_data.get('q_hat', 0.0))
                logger.info(f'✅ Conformal q_hat loaded: {self.conformal_scalars["ensemble"]}')
                self._load_report.append(('conformal', 'ok'))
            except Exception as e:
                logger.warning(f'⚠️  Conformal scalar failed to load: {e}')
                self.conformal_scalars['ensemble'] = 0.0
                self._load_report.append(('conformal', f'error: {e}'))

        loaded_ok = sum(1 for _, s in self._load_report if s == 'ok')
        total     = len(self._load_report)
        logger.info(f'Load complete: {loaded_ok}/{total} components ready.')
        return loaded_ok > 0  # At least one component must load for service to be usable

    # ── Prediction helpers ────────────────────────────────────────────────

    def predict_base_models(
        self,
        X_features: np.ndarray,
        tft_features: Dict[str, np.ndarray] = None
    ) -> np.ndarray:
        """
        Run all loaded base models and stack predictions column-wise.
        Columns: [ARIMA, Prophet, LightGBM, TFT]
        Unloaded models contribute a column of zeros (Ridge meta-learner
        will down-weight them to near-zero during ensemble).
        """
        n = len(X_features)
        preds = np.zeros((n, 4), dtype=np.float32)

        # ARIMA
        if 'arima' in self.models:
            try:
                preds[:, 0] = self.models['arima'].forecast(steps=n).values
            except Exception as e:
                logger.warning(f'ARIMA prediction failed: {e}')

        # Prophet
        if 'prophet' in self.models:
            try:
                future_df = pd.DataFrame({
                    'ds': pd.date_range(start=pd.Timestamp.today().normalize(), periods=n, freq='D')
                })
                prophet_out = self.models['prophet'].predict(future_df)
                preds[:, 1] = prophet_out['yhat'].values.astype(np.float32)
            except Exception as e:
                logger.warning(f'Prophet prediction failed: {e}')

        # LightGBM (ONNX)
        if 'lightgbm' in self.onnx_sessions:
            try:
                sess = self.onnx_sessions['lightgbm']
                input_name = sess.get_inputs()[0].name
                lgb_out = sess.run(None, {input_name: X_features.astype(np.float32)})
                preds[:, 2] = lgb_out[0].flatten().astype(np.float32)
            except Exception as e:
                logger.warning(f'LightGBM prediction failed: {e}')

        # TFT (ONNX)
        if 'tft' in self.onnx_sessions and tft_features is not None:
            try:
                sess = self.onnx_sessions['tft']
                ort_inputs = {
                    inp.name: tft_features[inp.name].astype(np.float32)
                    for inp in sess.get_inputs()
                    if inp.name in tft_features
                }
                tft_out = sess.run(None, ort_inputs)
                preds[:, 3] = tft_out[0].flatten().astype(np.float32)
            except Exception as e:
                logger.warning(f'TFT prediction failed: {e}')

        return preds

    def apply_conformal_bounds(
        self, point_prediction: np.ndarray
    ) -> Tuple[np.ndarray, np.ndarray]:
        """Applies symmetric conformal bounds → (P10, P90). P10 is floor-clamped at 0."""
        q_hat = float(self.conformal_scalars.get('ensemble', 0.0))
        p10 = np.maximum(point_prediction - q_hat, 0.0)
        p90 = point_prediction + q_hat
        return p10, p90

    def predict(
        self,
        X_features: np.ndarray,
        tft_features: Dict[str, np.ndarray] = None
    ) -> Dict[str, Any]:
        """
        End-to-end ensemble prediction.
        Returns dict with p10, p50, p90 lists plus diagnostics.

        Args:
            X_features  : 2-D array of shape (n_samples, n_features).
                          Must match the feature count your LightGBM was trained on (45 in production).
                          For smoke-testing use any n_features — only LightGBM/ARIMA/Prophet fire;
                          TFT needs tft_features dict.
            tft_features: dict mapping TFT ONNX input names to float32 arrays.
                          Pass None if TFT model is not loaded or you're doing a smoke test.
        """
        base_preds = self.predict_base_models(X_features, tft_features)

        if 'ensemble_ridge' not in self.models:
            logger.warning('Ridge meta-learner not loaded — using simple mean of base models.')
            p50_preds = np.mean(base_preds, axis=1)
        else:
            p50_preds = self.models['ensemble_ridge'].predict(base_preds)

        p10_preds, p90_preds = self.apply_conformal_bounds(p50_preds)

        return {
            'p10': p10_preds.tolist(),
            'p50': p50_preds.tolist(),
            'p90': p90_preds.tolist(),
            'base_model_predictions': base_preds.tolist(),
            'q_hat_applied': self.conformal_scalars.get('ensemble', 0.0),
        }

print('✅ ModelInferenceService defined.')

## 6. Load Models

In [ ]:
service = ModelInferenceService()
success = service.load_models()

print('\n── Load Report ──')
for component, status in service._load_report:
    icon = '✅' if status == 'ok' else ('⚠️ ' if status == 'missing' else '❌')
    print(f'  {icon} {component:20s} : {status}')

if not success:
    print('\n❌ No components loaded. Upload model files to Drive before continuing.')
else:
    loaded = sum(1 for _, s in service._load_report if s == 'ok')
    total  = len(service._load_report)
    print(f'\n✅ {loaded}/{total} components loaded — service is ready.')

## 7. Model Tests

In [ ]:
class ModelTester:
    """Validation tests for PoultryPulse model service."""

    def __init__(self, svc: ModelInferenceService):
        self.svc = svc
        self.test_results = []

    def _record(self, name: str, passed: int, total: int):
        rate = passed / total * 100 if total else 0.0
        self.test_results.append({'test': name, 'passed': passed, 'total': total, 'success_rate': rate})
        print(f'  Result: {passed}/{total} passed ({rate:.1f}%)')
        return self.test_results[-1]

    # ─────────────────────────────────────────────────────────────────────
    def test_model_loading(self):
        print('\n=== Test 1: Model Loading ===')
        checks = [
            ('arima',          'arima'          in self.svc.models),
            ('prophet',        'prophet'        in self.svc.models),
            ('lightgbm (ONNX)','lightgbm'       in self.svc.onnx_sessions),
            ('tft (ONNX)',     'tft'             in self.svc.onnx_sessions),
            ('ensemble_ridge', 'ensemble_ridge' in self.svc.models),
            ('conformal q_hat', self.svc.conformal_scalars.get('ensemble', -1) >= 0),
        ]
        for name, ok in checks:
            print(f'  {"✅" if ok else "⚠️ "} {name}')
        passed = sum(1 for _, ok in checks if ok)
        return self._record('Model Loading', passed, len(checks))

    # ─────────────────────────────────────────────────────────────────────
    def test_prediction_shape(self):
        print('\n=== Test 2: Prediction Shapes ===')
        n = 10
        # Use 45 features to match production; smoke-test with whatever is available
        n_feat = 45
        np.random.seed(42)  # Fixed seed for reproducibility
        X_test = np.random.randn(n, n_feat).astype(np.float32)
        passed = 0
        total  = 4
        try:
            result = self.svc.predict(X_test)

            for key in ('p10', 'p50', 'p90'):
                ok = len(result[key]) == n
                print(f'  {"✅" if ok else "❌"} {key} length == {n}: got {len(result[key])}')
                passed += int(ok)

            valid_intervals = all(
                lo <= mid <= hi
                for lo, mid, hi in zip(result['p10'], result['p50'], result['p90'])
            )
            print(f'  {"✅" if valid_intervals else "❌"} p10 ≤ p50 ≤ p90 for all samples')
            passed += int(valid_intervals)
        except Exception as e:
            print(f'  ❌ predict() raised: {e}')
        return self._record('Prediction Shapes', passed, total)

    # ─────────────────────────────────────────────────────────────────────
    def test_conformal_calibration(self):
        print('\n=== Test 3: Conformal Calibration ===')
        q = self.svc.conformal_scalars.get('ensemble', -1)
        passed = 0

        ok1 = q >= 0
        print(f'  {"✅" if ok1 else "❌"} q_hat is non-negative: {q}')
        passed += int(ok1)

        ok2 = q < 500  # Sanity upper bound for rupee/kg poultry prices
        print(f'  {"✅" if ok2 else "⚠️ "} q_hat < 500: {q}')
        passed += int(ok2)

        test_p = np.array([50.0], dtype=np.float32)
        p10, p90 = self.svc.apply_conformal_bounds(test_p)
        ok3 = float(p10[0]) >= 0.0
        print(f'  {"✅" if ok3 else "❌"} P10 lower bound ≥ 0: {p10[0]:.2f}')
        print(f'  ℹ️  P50=50 → P10={p10[0]:.2f}, P90={p90[0]:.2f}  (q_hat={q})')
        passed += int(ok3)

        return self._record('Conformal Calibration', passed, 3)

    # ─────────────────────────────────────────────────────────────────────
    def run_all_tests(self) -> list:
        print('\n' + '='*55)
        print('RUNNING ALL MODEL TESTS')
        print('='*55)
        self.test_model_loading()
        self.test_prediction_shape()
        self.test_conformal_calibration()
        print('\n' + '='*55)
        print('TEST SUMMARY')
        print('='*55)
        total_p = sum(r['passed'] for r in self.test_results)
        total_t = sum(r['total']  for r in self.test_results)
        for r in self.test_results:
            icon = '✅' if r['passed'] == r['total'] else ('⚠️ ' if r['passed'] > 0 else '❌')
            print(f"  {icon} {r['test']:30s} {r['passed']}/{r['total']} ({r['success_rate']:.1f}%)")
        overall = total_p / total_t * 100 if total_t else 0
        print(f"\n  Overall: {total_p}/{total_t} ({overall:.1f}%)")
        print('='*55)
        return self.test_results

print('✅ ModelTester defined.')

In [ ]:
tester = ModelTester(service)
test_results = tester.run_all_tests()

## 8. Sample Inference

In [ ]:
print('\n=== Sample Inference (7-day forecast) ===')

np.random.seed(42)  # Fixed seed — same input every run for reproducibility

n_samples  = 7   # 7-day forecast horizon
n_features = 45  # Match your production LightGBM feature count
             #   If your exported model uses fewer features, adjust this number.

X_sample = np.random.randn(n_samples, n_features).astype(np.float32)
print(f'Input shape : {X_sample.shape}  (samples × features)')
print(f'Forecasting : {n_samples} days starting {pd.Timestamp.today().date()}')

predictions = service.predict(X_sample)

results_df = pd.DataFrame({
    'Day'           : range(1, n_samples + 1),
    'Date'          : pd.date_range(start=pd.Timestamp.today().normalize(), periods=n_samples, freq='D').date,
    'P10 (₹/kg)'   : [round(v, 2) for v in predictions['p10']],
    'P50 (₹/kg)'   : [round(v, 2) for v in predictions['p50']],
    'P90 (₹/kg)'   : [round(v, 2) for v in predictions['p90']],
    'Interval (₹)' : [round(hi - lo, 2) for lo, hi in zip(predictions['p10'], predictions['p90'])],
})

print(f'\nq_hat applied : {predictions["q_hat_applied"]}')
print('\nPrediction Table:')
print(results_df.to_string(index=False))

## 9. API Business Logic Tests

In [ ]:
class APITestSimulator:
    """Python equivalents of the TypeScript farms.test.ts business-logic assertions."""

    def __init__(self):
        self.test_results = []

    def _run(self, name: str, fn):
        """Run a single test, catch AssertionError and log result independently."""
        print(f'\n=== {name} ===')
        try:
            fn()
            print(f'  ✅ {name} passed')
            self.test_results.append({'test': name, 'status': 'passed'})
        except AssertionError as e:
            print(f'  ❌ {name} FAILED: {e}')
            self.test_results.append({'test': name, 'status': f'failed: {e}'})
        except Exception as e:
            print(f'  ❌ {name} ERROR: {e}')
            self.test_results.append({'test': name, 'status': f'error: {e}'})

    # ── Individual tests ──────────────────────────────────────────────────

    def _test_fcr(self):
        """FCR = feed_kg / ((birds_placed - deaths) * avg_weight_g / 1000)"""
        feed_kg, birds, deaths, avg_g = 1000, 10000, 0, 2000
        fcr = feed_kg / ((birds - deaths) * avg_g / 1000)
        print(f'  Feed={feed_kg}kg  Birds={birds}  Deaths={deaths}  AvgWeight={avg_g}g')
        print(f'  FCR = {fcr}')
        assert abs(fcr - 0.05) < 1e-9, f'Expected 0.05, got {fcr}'

    def _test_mortality(self):
        deaths, birds = 100, 10000
        pct = (deaths / birds) * 100
        print(f'  Deaths={deaths}  Birds={birds}')
        print(f'  Mortality% = {pct}')
        assert abs(pct - 1.0) < 1e-9, f'Expected 1.0%, got {pct}'

    def _test_feed_per_bird(self):
        feed_kg, alive = 100, 10000
        g_per_bird = (feed_kg * 1000) / alive
        print(f'  Feed={feed_kg}kg  Alive={alive}')
        print(f'  Feed/bird = {g_per_bird}g')
        assert abs(g_per_bird - 10.0) < 1e-9, f'Expected 10g, got {g_per_bird}'

    def _test_avg_weight(self):
        sample_kg, sample_n = 200, 100
        avg_g = (sample_kg / sample_n) * 1000
        print(f'  SampleWeight={sample_kg}kg  SampleBirds={sample_n}')
        print(f'  AvgWeight = {avg_g}g')
        assert abs(avg_g - 2000.0) < 1e-9, f'Expected 2000g, got {avg_g}'

    # ── Runner ────────────────────────────────────────────────────────────

    def run_all_tests(self) -> list:
        print('\n' + '='*55)
        print('RUNNING API BUSINESS LOGIC TESTS')
        print('='*55)
        # Each test runs independently — one failure does NOT skip the rest
        self._run('FCR Calculation',         self._test_fcr)
        self._run('Mortality Percentage',    self._test_mortality)
        self._run('Feed Per Bird',           self._test_feed_per_bird)
        self._run('Average Weight',          self._test_avg_weight)

        print('\n' + '='*55)
        print('API TEST SUMMARY')
        print('='*55)
        passed = sum(1 for r in self.test_results if r['status'] == 'passed')
        for r in self.test_results:
            icon = '✅' if r['status'] == 'passed' else '❌'
            print(f"  {icon} {r['test']:30s} {r['status']}")
        print(f'\n  {passed}/{len(self.test_results)} tests passed')
        print('='*55)
        return self.test_results

print('✅ APITestSimulator defined.')

In [ ]:
api_tester  = APITestSimulator()
api_results = api_tester.run_all_tests()

## 10. Export Test Results

In [ ]:
# TESTS_DIR was created in Cell 4 (Imports). Writing there keeps results separate from models.
output_path = TESTS_DIR / f'test_results_{datetime.now().strftime("%Y%m%d_%H%M%S")}.json'

all_results = {
    'timestamp'     : datetime.now().isoformat(),
    'model_tests'   : tester.test_results,
    'api_tests'     : api_tester.test_results,
    'load_report'   : [{'component': k, 'status': s} for k, s in service._load_report],
}

with open(output_path, 'w') as f:
    json.dump(all_results, f, indent=2)

print(f'✅ Results saved to: {output_path}')
print('\nPreview:')
print(json.dumps(all_results, indent=2))

## 11. Summary

In [ ]:
print('\n' + '='*60)
print('POULTRYPULSE AI — COLAB NOTEBOOK SUMMARY')
print('='*60)

# Drive
print('  ✅ Google Drive mounted')

# Install
print('  ✅ Dependencies installed (run Cell 3 once; restart is automatic)')

# Models
loaded_ok = sum(1 for _, s in service._load_report if s == 'ok')
total_comp = len(service._load_report)
icon = '✅' if loaded_ok == total_comp else ('⚠️ ' if loaded_ok > 0 else '❌')
print(f'  {icon} Models: {loaded_ok}/{total_comp} components loaded')

# Model tests
mt_p = sum(r['passed'] for r in tester.test_results)
mt_t = sum(r['total']  for r in tester.test_results)
icon = '✅' if mt_p == mt_t else ('⚠️ ' if mt_p > 0 else '❌')
print(f'  {icon} Model tests: {mt_p}/{mt_t} passed')

# API tests
at_p = sum(1 for r in api_tester.test_results if r['status'] == 'passed')
at_t = len(api_tester.test_results)
icon = '✅' if at_p == at_t else ('⚠️ ' if at_p > 0 else '❌')
print(f'  {icon} API logic tests: {at_p}/{at_t} passed')

print(f'  ✅ Results exported to {TESTS_DIR.name}/')

print('\n  Next steps:')
print('  1. Review test_results_*.json in Drive')
print('  2. Any ⚠️  = model file missing (upload to Drive and re-run Cell 6)')
print('  3. Any ❌  = logic error (check the failing test output above)')
print('  4. Adjust n_features in Cell 8 to match your actual exported model input size')
print('='*60)